## Predicting Topics - Attacks using Natural Language Processing

### 1. Data Preprocessing (You guys can omit this code, Refer to the cleaned_data.csv) 

In [10]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

#Function to remove numbers out of the Text
def remove_numbers(text):
    # Remove numbers (digits) from the text
    return re.sub(r'\d+', '', text)

# Function to remove months from text
def remove_months(text):
    return re.sub(month_pattern, '', text, flags=re.IGNORECASE)

df_gender = pd.read_csv('Data/gender_Apr25-1.csv')
df_health = pd.read_csv('Data/healthworkers_Apr18.csv')
df_journalists = pd.read_csv('Data/journalists_Apr18.csv')
df_peacekeeping = pd.read_excel('Data/peacekeeping_2025-05-02.xlsx')

df_gender = df_gender[['notes']]
df_gender[['target']] = 'gender'

df_health = df_health[['notes']]
df_health[['target']] = 'health' 

df_journalists = df_journalists[['notes']]
df_journalists[['target']] = 'journalists' 


df_peacekeeping[['notes']] = df_peacekeeping[['NOTES']]
df_peacekeeping = df_peacekeeping[['notes']]
df_peacekeeping[['target']] = 'peacekepers'

df_merged = pd.concat([df_gender, df_health, df_journalists, df_peacekeeping], ignore_index=True) 
df_merged['clean_text'] = df_merged['notes'].apply(remove_numbers)

#Cleaning months out of the text
months = [
    "january", "february", "march", "april", "may", "june", 
    "july", "august", "september", "october", "november", "december"
]

# Create a regex pattern to match month names (case insensitive)
month_pattern = r'\b(?:' + '|'.join(months) + r')\b'
# Apply the function to the 'text' column
df_merged['clean_text'] = df_merged['clean_text'].apply(remove_months)

#Remove stop words using NLTK
stop_words = set(stopwords.words('english'))

# Function to remove stop words
def remove_stopwords(text):
    tokens = word_tokenize(text.lower())
    filtered = [word for word in tokens if word.lower() not in stop_words and word.isalpha()]  # removes punctuation too
    return " ".join(filtered)

# Apply the function to the text column
df_merged['clean_text'] = df_merged['clean_text'].apply(remove_stopwords)

df_merged.to_csv('data_cleaned.csv', index = False)

df_merged

,notes,target,clean_text
0,"On 25 April 2025, in Capilla del Monte (Cordob...",gender,capilla del monte cordoba large group people i...
1,"Around 25 April 2025 (as reported), in Salvado...",gender,around reported salvador bahia cv members shot...
2,"On 25 April 2025, about 200 Israelis from the ...",gender,israelis shift movement protested jerusalem ju...
3,"On 25 April 2025, in Leon de los Aldama, Guana...",gender,leon de los aldama guanajuato woman shot dead ...
4,"On 25 April 2025, in Sabanas de Xalostoc, Vera...",gender,sabanas de xalostoc veracruz armed individuals...
...,...,...,...
152120,Houthi forces reportedly fired a Katyusha rock...,peacekepers,houthi forces reportedly fired katyusha rocket...
152121,"On 2 March 2019, anti-Houthi forces reportedly...",peacekepers,forces reportedly opened fire convoy head unmh...
152122,"On 26 February 2019, pro-Houthi forces shelled...",peacekepers,forces shelled unmha convoy near hospital area...
152123,Houthi forces reportedly fired at the UN team ...,peacekepers,houthi forces reportedly fired un team observe...


### 2. Running a preliminary Neural Network

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from sklearn.preprocessing import LabelEncoder

tokenizer = Tokenizer(num_words=5000)  # Limit the vocabulary size to 5000 words
tokenizer.fit_on_texts(df_merged['clean_text'])
X = tokenizer.texts_to_sequences(df_merged['clean_text'])

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(df_merged['target'])

X_pad = pad_sequences(X, padding='post', maxlen=10)
y = to_categorical(y_int, num_classes=4)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42, stratify=y)

# Define the model
model = Sequential([
    Embedding(input_dim=5000, output_dim=64, input_length=10),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')  # Assuming 4 target classes
])

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, 
                    epochs=10, 
                    batch_size=32, 
                    validation_data=(X_test, y_test))

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

/Users/felipe_q/Desktop/AI_models/MACHINE_LEARNING/mlenv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8377 - loss: 0.4799 - val_accuracy: 0.8888 - val_loss: 0.3156
Epoch 2/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.9016 - loss: 0.2721 - val_accuracy: 0.8872 - val_loss: 0.3124
Epoch 3/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.9079 - loss: 0.2523 - val_accuracy: 0.8880 - val_loss: 0.3137
Epoch 4/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.9103 - loss: 0.2368 - val_accuracy: 0.8864 - val_loss: 0.3183
Epoch 5/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9142 - loss: 0.2279 - val_accuracy: 0.8875 - val_loss: 0.3267
Epoch 6/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.9158 - loss: 0.2198 - val_accuracy: 0.8846 - val_loss: 0.3332
Epoch 7/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9199 - loss: 0.2098 - val_accuracy: 0.8834 - val_loss: 0.3412
Epoch 8/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.9240 - loss: 0

In [12]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_probs = model.predict(X_test)

# Step 2: Convert probabilities and one-hot true labels to class indices
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Step 3: Generate confusion matrix and classification report
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred)

# Display results
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

951/951 ━━━━━━━━━━━━━━━━━━━━ 1s 468us/step
Confusion Matrix:
 [[15400   946   438    40]
 [ 1251  7335   181    15]
 [  613   178  3295    22]
 [   52    23    10   626]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.92      0.90     16824
           1       0.86      0.84      0.85      8782
           2       0.84      0.80      0.82      4108
           3       0.89      0.88      0.89       711

    accuracy                           0.88     30425
   macro avg       0.87      0.86      0.86     30425
weighted avg       0.88      0.88      0.88     30425

